In [ ]:
import json
import pandas as pd
import numpy as np
import random
import re
from tqdm import tqdm
import unicodedata
import spacy
import numpy as np
from scipy.stats import pointbiserialr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 67

In [ ]:
#student introduction posts
#ground_truth_posts = pd.read_csv('PERSONALITY_DATASET_2024.csv') #not available due to IRB
#posts_text = ground_truth_posts['introduction_post']
#posts_sample = ground_truth_posts.filter(['openness','conscientiousness','extroversion', 'agreeableness', 'neuroticism'], axis=1).astype(float)

#essays - full set
ground_truth_essays = pd.read_csv('essays.csv', encoding='ISO-8859-1')
traits = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]
mapping = {'y': 1, 'n': 0}
personality_traits = ground_truth_essays[[
    'cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN'
]].replace(mapping)
personality_traits.columns = traits
personality_traits = personality_traits.astype(int)
personality_traits['profile'] = (
    personality_traits.astype(str).agg('-'.join, axis=1)
)
stratified_essay_sample = personality_traits

full_selected_texts = ground_truth_essays.loc[stratified_essay_sample.index, 'TEXT'].tolist()
full_personality_traits = stratified_essay_sample.reset_index(drop=True)

#essays - subset
traits = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]

mapping = {'y': 1, 'n': 0}

personality_traits = ground_truth_essays[[
    'cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN'
]].replace(mapping)

personality_traits.columns = traits
subset_personality_traits = personality_traits.astype(int)

subset_personality_traits['profile'] = (
    subset_personality_traits.astype(str).agg('-'.join, axis=1)
)

sampled_indices, _ = train_test_split(
    subset_personality_traits.index,
    train_size=226,
    stratify=subset_personality_traits['profile'],
    random_state=SEED
)


subset_selected_texts = ground_truth_essays.loc[
    sampled_indices, 'TEXT'
].tolist()

subset_personality_traits = (
    subset_personality_traits
    .loc[sampled_indices, traits]
    .reset_index(drop=True)
)

#facebook
ground_truth_FB = pd.read_csv('mypersonality_final.csv', encoding='ISO-8859-1')

In [ ]:
#student introduction post dataset midpoint procedure for conversation to binary
'''
posts_sample['openness'] = posts_sample['openness'].apply(
    lambda x: 0 if x <= 3 
    else 1
)

posts_sample['conscientiousness'] = posts_sample['conscientiousness'].apply(
    lambda x: 0 if x <= 3 
    else 1
)

posts_sample['extroversion'] = posts_sample['extroversion'].apply(
    lambda x: 0 if x <= 3 
    else 1
)

posts_sample['agreeableness'] = posts_sample['agreeableness'].apply(
    lambda x: 0 if x <= 3 
    else 1
)

posts_sample['neuroticism'] = posts_sample['neuroticism'].apply(
    lambda x: 0 if x <= 3 
    else 1
)
'''

pass

In [ ]:
#Facebook MILP to create n=164 subset

# Implements the maximum-size approximately balanced sampling
# procedure described in the paper's Datasets/Facebook Statuses section.
def make_balanced_user_sample(
    df,
    target_n=None,
    tol=0.05,
    seed=SEED,
    id_col='#AUTHID',
    label_cols=('cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN')
):
    label_cols = list(label_cols)

    one_per_user = (
        df
        .groupby(id_col, group_keys=False)
        .sample(n=1, random_state=SEED)
        .copy()
    )

    one_per_user['_profile'] = (
        one_per_user[label_cols]
        .astype(str)
        .agg('-'.join, axis=1)
    )

    profile_counts = one_per_user['_profile'].value_counts().sort_index()
    profiles = profile_counts.index.tolist()
    caps = profile_counts.to_numpy()
    P = len(profiles)

    A = np.array([
        [1 if prof.split('-')[j] == 'y' else 0 for prof in profiles]
        for j in range(len(label_cols))
    ])

    c = -np.ones(P)

    constraints = []
    lb = []
    ub = []

    if target_n is not None:
        constraints.append(np.ones(P))
        lb.append(target_n)
        ub.append(target_n)

    lower_prop = 0.5 - tol
    upper_prop = 0.5 + tol

    total_row = np.ones(P)

    for j in range(len(label_cols)):
        y_row = A[j]

        constraints.append(y_row - lower_prop * total_row)
        lb.append(0)
        ub.append(np.inf)

        constraints.append(y_row - upper_prop * total_row)
        lb.append(-np.inf)
        ub.append(0)

    M = np.vstack(constraints)

    result = milp(
        c=c,
        integrality=np.ones(P),
        bounds=Bounds(np.zeros(P), caps),
        constraints=LinearConstraint(M, np.array(lb), np.array(ub))
    )

    if not result.success:
        raise RuntimeError(result.message)

    alloc = pd.Series(
        np.rint(result.x).astype(int),
        index=profiles,
        name='n_sampled'
    )

    sampled_parts = []
    for prof, n in alloc[alloc > 0].items():
        group = one_per_user[one_per_user['_profile'] == prof]
        sampled_parts.append(group.sample(n=int(n), random_state=SEED))

    sample = (
        pd.concat(sampled_parts)
        .sample(frac=1, random_state=SEED)
        .drop(columns=['_profile'])
        .copy()
    )

    return sample, alloc

facebook_balanced, facebook_alloc = make_balanced_user_sample(
    ground_truth_FB
)

label_cols = ['cEXT', 'cNEU', 'cAGR', 'cCON', 'cOPN']

trait_names = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]

stratified_FB_sample = facebook_balanced.copy()
stratified_FB_sample['original_index'] = stratified_FB_sample.index

selected_FB_texts = ground_truth_FB.loc[
    stratified_FB_sample['original_index'],
    'STATUS'
].tolist()

personality_traitsFB = ground_truth_FB.loc[
    stratified_FB_sample['original_index'],
    label_cols
].copy()

personality_traitsFB.columns = trait_names

personality_traitsFB = personality_traitsFB.replace({'y': 1, 'n': 0}).astype(int)

stratified_FB_sample = stratified_FB_sample.reset_index(drop=True)
personality_traitsFB = personality_traitsFB.reset_index(drop=True)

In [ ]:
# essay full set distrobution statistics
traits = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]

print("Rows:", len(full_personality_traits))
print("Unique users:", full_personality_traits.shape[0])

for c in traits:
    print(c)
    print(full_personality_traits[c].value_counts(normalize=True))

score_cols = traits
label_cols = traits

summary = pd.DataFrame({
    col: full_personality_traits[col].value_counts(normalize=True)
    for col in label_cols
}).T

print(summary)

In [ ]:
# essay subset distrobution statistics
traits = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]

print("Rows:", len(subset_personality_traits))
print("Unique users:", subset_personality_traits.shape[0])

for c in traits:
    print(c)
    print(subset_personality_traits[c].value_counts(normalize=True))

score_cols = traits
label_cols = traits

summary = pd.DataFrame({
    col: subset_personality_traits[col].value_counts(normalize=True)
    for col in label_cols
}).T

print(summary)

In [ ]:
# facebook subset distrobution statistics
traits = [
    'extroversion',
    'neuroticism',
    'agreeableness',
    'conscientiousness',
    'openness'
]

print("Rows:", len(personality_traitsFB))
print("Unique users:", personality_traitsFB.shape[0])

for c in traits:
    print(c)
    print(personality_traitsFB[c].value_counts(normalize=True))

score_cols = traits
label_cols = traits

summary = pd.DataFrame({
    col: personality_traitsFB[col].value_counts(normalize=True)
    for col in label_cols
}).T

print(summary)

In [ ]:
#Mask PII from student introduction posts dataset
'''
EMAIL_PATTERN  = re.compile(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b")
URL_PATTERN    = re.compile(r"(?:https?://|www\.)\S+")
PHONE_PATTERN  = re.compile(r"(?:\+?\d{1,3}[\s.\-]?)?\(?\d{3}\)?[\s.\-]?\d{3}[\s.\-]?\d{4}\b")
HANDLE_PATTERN = re.compile(r"(?<![A-Za-z0-9])@[A-Za-z0-9_]{2,}\b")

NAME_LABELS = {"PERSON", "FAC"} 

def strip_pii(text):
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = URL_PATTERN.sub("[URL]", text)
    text = PHONE_PATTERN.sub("[PHONE]", text)
    text = HANDLE_PATTERN.sub("[HANDLE]", text)
    
    doc = nlp(text)
    result = text
    for ent in reversed(doc.ents):
        if ent.label_ in NAME_LABELS:
            result = result[:ent.start_char] + "[NAME]" + result[ent.end_char:]
    return result

posts_text = [strip_pii(p) for p in posts_text]
'''

pass

In [ ]:
pd.DataFrame({"texts": full_selected_texts}).to_csv('ESSAYS_fullset_text.csv', index=True)
full_personality_traits.to_csv('ESSAYS_fullset_values.csv', index=False)

pd.DataFrame({"texts": subset_selected_texts}).to_csv('ESSAYS_subset_text.csv', index=True)
subset_personality_traits.to_csv('ESSAYS_subset_values.csv', index=False)

In [ ]:
stratified_FB_sample.to_csv('FACEBOOK_subset_text.csv', index=False)
personality_traitsFB.to_csv('FACEBOOK_subset_values.csv', index=False)

In [ ]:
#posts_text.to_csv('POSTS_subset_text.csv', index=False)
#posts_sample.to_csv('POSTS_subset_values.csv', index=False)